In [1]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="ESG_DB",
    user="postgres",
    password=os.environ["DB_PASSWORD"]
)

print("Connected to PostgreSQL!")
conn.close()

Connected to PostgreSQL!


In [1]:
# load libraries
import requests
import os
import time

In [10]:
companies = [
    # US Banks
    {"ticker": "JPM",  "cik": "0000019617", "name": "JPMorgan Chase"},
    {"ticker": "BAC",  "cik": "0000070858", "name": "Bank of America"},
    {"ticker": "WFC",  "cik": "0000072971", "name": "Wells Fargo"},
    {"ticker": "GS",   "cik": "0000886982", "name": "Goldman Sachs"},
    {"ticker": "MS",   "cik": "0000895421", "name": "Morgan Stanley"},
    {"ticker": "C",    "cik": "0000831001", "name": "Citigroup"},
    {"ticker": "USB",  "cik": "0000036104", "name": "US Bancorp"},
    {"ticker": "PNC",  "cik": "0000713676", "name": "PNC Financial"},
    {"ticker": "AXP",  "cik": "0000004962", "name": "American Express"},
    {"ticker": "BLK",  "cik": "0001364742", "name": "BlackRock"},
    {"ticker": "COF",  "cik": "0000927628", "name": "Capital One"},
    {"ticker": "TFC",  "cik": "0000092122", "name": "Truist Financial"},
    {"ticker": "BK",   "cik": "0001390777", "name": "Bank of New York Mellon"},
    {"ticker": "STT",  "cik": "0000093751", "name": "State Street"},
    # European Banks
    {"ticker": "HSBC", "cik": "0000083246", "name": "HSBC"},
    {"ticker": "BCS",  "cik": "0000312070", "name": "Barclays"},
    {"ticker": "DB",   "cik": "0001159508", "name": "Deutsche Bank"},
    {"ticker": "UBS",  "cik": "0001114446", "name": "UBS"},
    {"ticker": "ING",  "cik": "0001160106", "name": "ING Group"},
    {"ticker": "SAN",  "cik": "0000891478", "name": "Santander"},
    {"ticker": "LYG",  "cik": "0000763901", "name": "Lloyds Banking Group"},
]

In [7]:
os.makedirs("data", exist_ok=True)

headers = {"User-Agent": "bianca.cadisch@usi.ch"}

def get_latest_10k(cik):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        return None, None
    
    data = response.json()
    filings = data["filings"]["recent"]
    forms = filings["form"]
    dates = filings["filingDate"]
    accession_numbers = filings["accessionNumber"]
    
    for i, form in enumerate(forms):
        if form == "10-K" or form == "20-F":
            return accession_numbers[i], dates[i]
    
    return None, None

In [8]:
def download_report(company):
    ticker = company["ticker"]
    cik = company["cik"]
    name = company["name"]
    
    print(f"Fetching {name} ({ticker})...")
    
    accession, date = get_latest_10k(cik)
    
    if not accession:
        print(f"  Could not find filing for {ticker}")
        return
    
    save_path = f"data/{ticker}_10K_{date}.txt"
    with open(save_path, "w") as f:
        f.write(f"Company: {name}\n")
        f.write(f"Ticker: {ticker}\n")
        f.write(f"CIK: {cik}\n")
        f.write(f"Filing date: {date}\n")
        f.write(f"Accession number: {accession}\n")
    
    print(f"  Saved {ticker} — filed {date}")
    time.sleep(0.5)

In [9]:
for company in companies:
    download_report(company)

Fetching JPMorgan Chase (JPM)...
  Saved JPM — filed 2026-02-13
Fetching Bank of America (BAC)...
  Saved BAC — filed 2026-02-25
Fetching Wells Fargo (WFC)...
  Saved WFC — filed 2026-02-24
Fetching Goldman Sachs (GS)...
  Saved GS — filed 2026-02-25
Fetching Morgan Stanley (MS)...
  Saved MS — filed 2026-02-19
Fetching Citigroup (C)...
  Saved C — filed 2026-02-20
Fetching US Bancorp (USB)...
  Saved USB — filed 2026-02-23
Fetching PNC Financial (PNC)...
  Saved PNC — filed 2026-02-20
Fetching Charles Schwab (SCHW)...
  Saved SCHW — filed 2022-03-31
Fetching American Express (AXP)...
  Saved AXP — filed 2026-02-06
Fetching BlackRock (BLK)...
  Saved BLK — filed 2024-02-23
Fetching Capital One (COF)...
  Saved COF — filed 2026-02-19
Fetching Truist Financial (TFC)...
  Saved TFC — filed 2026-02-19
Fetching Bank of New York Mellon (BK)...
  Saved BK — filed 2026-02-25
Fetching State Street (STT)...
  Saved STT — filed 2026-02-19
Fetching HSBC (HSBC)...
  Saved HSBC — filed 2026-02-25
Fe

In [11]:
def get_report_text(ticker, cik, accession):
    """Download the actual text of the filing"""
    
    # Format accession number for URL
    accession_clean = accession.replace("-", "")
    
    # Get the filing index
    index_url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=10-K&dateb=&owner=include&count=1"
    
    # Get list of documents in this filing
    docs_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(docs_url, headers=headers)
    data = response.json()
    
    # Build the direct URL to the filing text
    accession_formatted = f"{accession[:10]}/{accession[10:12]}/{accession[12:]}"
    filing_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_clean}/{accession}-index.htm"
    
    response = requests.get(filing_url, headers=headers)
    return response.text

In [12]:
# Test with just JPMorgan first
test_company = {"ticker": "JPM", "cik": "0000019617", "name": "JPMorgan Chase"}

# Read the accession number we already saved
with open("data/JPM_10K_2026-02-13.txt", "r") as f:
    lines = f.readlines()
    for line in lines:
        if "Accession" in line:
            accession = line.split(": ")[1].strip()

print(f"Accession number: {accession}")
text = get_report_text(test_company["ticker"], test_company["cik"], accession)
print(f"Got {len(text)} characters of text")
print("\nFirst 500 characters:")
print(text[:500])

Accession number: 0001628280-26-008131
Got 18637 characters of text

First 500 characters:
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
<meta http-equiv="Last-Modified" content="Fri, 13 Feb 2026 21:20:00 GMT" />
<title>EDGAR Filing Documents for 0001628280-26-008131</title>
<link  rel="stylesheet" href="/edgar/search/global/css/bootstrap/bootstrap.min.css" type="text/css" />
<link rel="stylesheet" type="text/


In [13]:
from bs4 import BeautifulSoup

# Parse the index page to find the actual 10-K document link
soup = BeautifulSoup(text, "html.parser")

# Find all links in the filing index
links = soup.find_all("a", href=True)

# Look for the main document (usually a .htm file)
for link in links:
    href = link["href"]
    if href.endswith(".htm") and "R" not in href:
        print(href)

/index.htm
/ix?doc=/Archives/edgar/data/19617/000162828026008131/jpm-20251231.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit46.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit102.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit1017.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit19.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit21.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit222.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit23.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit311.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit312.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit32.htm
/Archives/edgar/data/19617/000162828026008131/corp10k2025exhibit97.htm


In [14]:
# Download the actual 10-K document
main_doc_url = "https://www.sec.gov/Archives/edgar/data/19617/000162828026008131/jpm-20251231.htm"

response = requests.get(main_doc_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Extract just the text, no HTML tags
clean_text = soup.get_text()

print(f"Got {len(clean_text)} characters of text")
print("\nFirst 500 characters:")
print(clean_text[:500])

Got 1380916 characters of text

First 500 characters:





jpm-202512310000019617FALSE2025FYtruefalseJune 30, 2026230June 30, 2026228June 30, 2026229August 7, 2026270June 30, 2026230June 30, 2026230June 30, 2026228June 30, 2026228June 30, 2026229June 30, 2026229http://fasb.org/us-gaap/2025#PrincipalTransactionsRevenuehttp://fasb.org/us-gaap/2025#PrincipalTransactionsRevenuehttp://fasb.org/us-gaap/2025#PrincipalTransactionsRevenuehttp://fasb.org/us-gaap/2025#TradingLiabilitieshttp://fasb.org/us-gaap/2025#TradingLiabilitiesP5Yhttp://fasb.org/us-gaap/


In [15]:
# Search for ESG related content
keywords = ["carbon", "emissions", "sustainability", "governance", "diversity"]

for keyword in keywords:
    # Find where this keyword appears
    index = clean_text.lower().find(keyword)
    if index != -1:
        print(f"\n--- Found '{keyword}' at position {index} ---")
        print(clean_text[index:index+300])


--- Found 'carbon' at position 369884 ---
carbon taxes and the adoption of new technologies to support lower-carbon operations, may increase compliance and operational costs, contribute to commodity price volatility and impact the profitability of clients and customers that are adapting to a low-carbon economy. Any of these impacts could ha

--- Found 'sustainability' at position 262071 ---
Sustainability:Policymakers in the U.K. and the EU continue to implement and refine sustainability-related initiatives and disclosure requirements. The Corporate Sustainability Reporting Directive (“CSRD”) replaced and significantly expanded the scope and content of certain EU ESG reporting requirem

--- Found 'governance' at position 229795 ---
Governance.37Item 11.Executive Compensation.38Item 12.Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters.38Item 13.Certain Relationships and Related Transactions, and Director Independence.38Item 14.Principal Acco

In [16]:
# Save the clean text to our data folder
with open("data/JPM_fulltext.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)

print("Saved JPM full text!")
print(f"File size: {len(clean_text)} characters")

Saved JPM full text!
File size: 1380916 characters


In [17]:
def split_into_chunks(text, chunk_size=500):
    """Split text into chunks of roughly chunk_size words"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    
    return chunks

# Test on JPMorgan
chunks = split_into_chunks(clean_text)

print(f"Total chunks: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[10])

Total chunks: 322

Example chunk:
Securities LLC are registered with the SEC as “security-based swap dealers.” As a result, these entities are subject to a comprehensive regulatory framework applicable to their swap or security-based swap activities, including capital requirements, rules requiring the collateralization of uncleared swaps and security-based swaps, rules regarding segregation of counterparty collateral, business conduct and documentation standards, rules requiring the central 5Part Iclearing of standardized over-the-counter (“OTC”) derivatives, requirements that certain standardized OTC swaps be traded on regulated trading venues, record-keeping and reporting obligations, and anti-fraud and anti-manipulation requirements. Similar requirements have also been established in the European Union (“EU”) under the European Market Infrastructure Regulation (“EMIR”) and the Markets in Financial Instruments Directive (“MiFID II”), as well as in the U.K. and other jurisdictions aro

In [18]:
import json

# Save chunks to a JSON file
chunks_data = {
    "ticker": "JPM",
    "name": "JPMorgan Chase",
    "total_chunks": len(chunks),
    "chunks": chunks
}

with open("data/JPM_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(chunks)} chunks for JPMorgan!")

Saved 322 chunks for JPMorgan!


In [19]:
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# Find a chunk that mentions carbon/ESG
esg_chunk = None
for chunk in chunks:
    if "carbon" in chunk.lower() or "emissions" in chunk.lower():
        esg_chunk = chunk
        break

print("Found ESG chunk:")
print(esg_chunk[:300])

Found ESG chunk:
as a result of impacts on its clients and customers.Both physical risks and transition risks associated with climate change could negatively impact JPMorganChase and its clients and customers. Physical risks include the increased frequency or severity of acute weather events and shifting climate pat


In [23]:
# Find all chunks mentioning environmental topics
environmental_chunks = []

for i, chunk in enumerate(chunks):
    chunk_lower = chunk.lower()
    if any(word in chunk_lower for word in ["carbon", "emissions", "climate", "sustainability", "renewable", "net zero"]):
        environmental_chunks.append(chunk)

print(f"Found {len(environmental_chunks)} environmental chunks out of {len(chunks)} total")
print("\nFirst relevant chunk preview:")
print(environmental_chunks[0][:300])

Found 16 environmental chunks out of 322 total

First relevant chunk preview:
and liabilities without the approval of the institution’s creditors. Prompt corrective action. The Federal Deposit Insurance Corporation Improvement Act of 1991 requires the relevant federal banking regulator to take “prompt corrective action” with respect to a depository institution if that institu


In [24]:
# Combine all environmental chunks into one text
combined_environmental = "\n\n---\n\n".join(environmental_chunks)

print(f"Total characters being sent to Gemini: {len(combined_environmental)}")

Total characters being sent to Gemini: 58617


In [26]:
from pydantic import BaseModel, Field

class ESGScore(BaseModel):
    score: int = Field(description="Score from 0 to 10")
    justification: str = Field(description="One sentence explaining the score")
    evidence_quote: str = Field(description="Short quote from the text that supports the score")

print("ESGScore loaded!")

ESGScore loaded!


In [29]:
response = client.models.generate_content(
    model="gemini-2-flash",
    contents=f"""You are an ESG analyst specialising in banks.
    
Based ONLY on the text below from JPMorgan Chase's annual report, rate their 
performance on ENVIRONMENTAL factors (carbon emissions, climate risk, 
sustainability commitments, renewable energy) on a scale of 0-10.

A score of 0 means no environmental effort mentioned at all.
A score of 10 means world leading environmental commitments with concrete targets.

Text from annual report:
{combined_environmental}

Respond in JSON with these fields: score (int), justification (str), evidence_quote (str)""",
    config={"response_mime_type": "application/json",
            "response_schema": ESGScore}
)

result = ESGScore.model_validate_json(response.text)
print(f"Environmental Score: {result.score}/10")
print(f"Justification: {result.justification}")
print(f"Evidence: {result.evidence_quote}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-2-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}